# Session 1: a secret message
**A. Real-world problem.** You want to pass a note in class so that only your friend can
read it. A very old trick, used by Julius Caesar, is to *shift* every letter along the
alphabet by a fixed number. This is a **toy cipher**: it is perfect for learning and
useless for protecting anything real. Today you encrypt and decrypt by hand and in code,
then discover why anyone can break it in a minute.

**Plan.** (1) shift letters with modular arithmetic, (2) decrypt, (3) try all shifts,
(4) let the computer pick the readable one.

In [ ]:
import pathlib
import string
import time

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
ALPHABET = string.ascii_uppercase           # A..Z, 26 letters
MESSAGE = "MEET ME AT THE OLD GATE AT NOON"  # the secret note used in the app
COMMON = "ETAOINSHR"                         # the nine most common English letters

## Step 1: shift a letter
Number the letters A=0, B=1, ..., Z=25. Encrypting with key k means: new number =
(old number + k) mod 26. "mod 26" means: after Z, wrap round to A.

Check by hand with k = 3: H is 7, so 7 + 3 = 10, which is K. E is 4, so 7 -> K, 4 -> H.
"HELLO" becomes "KHOOR". Y is 24, and 24 + 3 = 27, which wraps to 27 - 26 = 1, so Y -> B.

In [ ]:
def shift_text(text, k):
    """Shift every letter by k places (k can be negative or larger than 25).
    Letters keep their case; spaces and punctuation are left alone."""
    out = []
    for ch in text:
        if ch.isalpha() and ch.isascii():
            base = ord("A") if ch.isupper() else ord("a")
            out.append(chr((ord(ch) - base + k) % 26 + base))
        else:
            out.append(ch)
    return "".join(out)


def caesar_encrypt(text, k):
    return shift_text(text, k)


def caesar_decrypt(text, k):
    return shift_text(text, -k)

## Step 2: try every shift
Shift 0 and shift 26 change nothing, so there are only **25** useful keys. An attacker
does not need to be clever: write down all 25 shifts and read the one that makes sense.

In [ ]:
def all_shifts(ciphertext):
    """Return [(k, text), ...] for k = 1..25: the ciphertext decrypted with each key."""
    return [(k, caesar_decrypt(ciphertext, k)) for k in range(1, 26)]

## Step 3: let the computer read for us
English uses some letters much more than others. The nine letters E T A O I N S H R make
up most of normal English text, while a random mix of letters would give only 9/26,
about 35 percent. A *readability score* = share of letters that are in COMMON.
The right shift gives a high score; wrong shifts give low ones.

In [ ]:
def readability(text):
    """Fraction of the letters of text that are among the nine common letters."""
    letters = [c for c in text.upper() if c in ALPHABET]
    if not letters:
        return 0.0
    return sum(c in COMMON for c in letters) / len(letters)


def score_by_guess(ciphertext):
    """Readability of the ciphertext decrypted with each guess 0..25 (26 numbers)."""
    return [readability(caesar_decrypt(ciphertext, g)) for g in range(26)]


def best_shift(ciphertext):
    """The guess (0..25) whose decryption looks most like English."""
    scores = score_by_guess(ciphertext)
    return max(range(26), key=lambda g: scores[g])


def time_to_try_all(ciphertext):
    """Seconds the computer needs to decrypt with all 25 keys and score them."""
    t0 = time.perf_counter()
    best_shift(ciphertext)
    return time.perf_counter() - t0

## Step 4: see it
Left: the two alphabet rows (plain on top, cipher below) for key 3. Right: the
readability score for each of the 26 guesses on a message encrypted with key 3. Only one
guess stands out: the right one.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    key = 3
    ct = caesar_encrypt(MESSAGE, key)
    print("message   :", MESSAGE)
    print("ciphertext:", ct)
    print("HELLO with key 3 ->", caesar_encrypt("HELLO", 3))
    print("best guess:", best_shift(ct), " time to try all keys: %.5f s" % time_to_try_all(ct))
    for k, t in all_shifts(ct)[:5]:
        print(f"  key {k:2d}: {t}")
    sc = score_by_guess(ct)
    print("score at the right guess: %.3f ; best wrong guess: %.3f" % (sc[key], max(s for g, s in enumerate(sc) if g != key)))

    fig, ax = plt.subplots(figsize=(8, 2.6))
    for i, ch in enumerate(ALPHABET):
        ax.text(i, 1, ch, ha="center", va="center", fontsize=12)
        ax.text(i, 0, ALPHABET[(i + key) % 26], ha="center", va="center", fontsize=12, color="tab:red")
        ax.plot([i, i], [0.25, 0.75], color="gray", lw=0.6)
    ax.text(-1.2, 1, "plain", ha="right", va="center")
    ax.text(-1.2, 0, "cipher", ha="right", va="center", color="tab:red")
    ax.set_xlim(-3.5, 25.8)
    ax.set_ylim(-0.5, 1.5)
    ax.axis("off")
    ax.set_title("Caesar cipher with key 3: every letter moves 3 places along")
    fig.tight_layout()
    fig.savefig(OUT / "caesar_wheel.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6.5, 4))
    ax2.bar(range(26), [100 * s for s in sc], color=["tab:red" if g == key else "tab:gray" for g in range(26)])
    ax2.set_xlabel("guessed key (shift used to decrypt)")
    ax2.set_ylabel("readability score (% common letters)")
    ax2.set_title("Trying all shifts: only the right key reads as English")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "try_all_shifts.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint. Remember: Caesar is a toy. Real systems use vetted libraries, never home-made
ciphers.

## Coding tasks
1. **Easy.** Use `caesar_encrypt` to encrypt your own name with key 7, then decrypt it with `caesar_decrypt`. Predict the first letter by hand before you run it.
2. **Stretch.** Write `rot13(text)` (key 13) and check that applying it twice gives back the original. Explain why only key 13 does this.
3. **Explain.** Explain why the readability score might pick the wrong key for the three-letter message CAT, and what you could change (a longer message, or a better score).

## Assessment
- **A. Conceptual.** Why can anyone break a Caesar cipher, even without knowing the key? Does it help to keep the method secret?
- **B. Numerical.** Encrypt the word GO with key 20. Show the wrap-around, then check by decrypting.
- **C. Graph and simulation interpretation.** In the graph of readability against guess, why is one bar tall and the others low? Why can the gap be small for a very short message?
- **D. Design and optimisation challenge.** Design: you must send a note that a classmate cannot read in the time they have, using only the ideas of this session. List two changes you could make and what each costs.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("hand calculation: HELLO with key 3 is KHOOR", L.caesar_encrypt("HELLO", 3) == "KHOOR"))
    checks.append(("wrap-around: Y shifted by 3 is B", L.caesar_encrypt("Y", 3) == "B"))
    ct = L.caesar_encrypt("SEE YOU AT NOON", 5)
    checks.append(("worked example: SEE YOU AT NOON with key 5 is XJJ DTZ FY STTS", ct == "XJJ DTZ FY STTS"))
    checks.append(("worked example: decrypting it with key 5 returns the note", L.caesar_decrypt(ct, 5) == "SEE YOU AT NOON"))
    ok = all(L.caesar_decrypt(L.caesar_encrypt(L.MESSAGE, k), k) == L.MESSAGE for k in range(-30, 60))
    checks.append(("decrypt undoes encrypt for every key from -30 to 59", ok))
    checks.append(("key 0 and key 26 change nothing (so only 25 useful keys)",
                   L.caesar_encrypt(L.MESSAGE, 0) == L.MESSAGE and L.caesar_encrypt(L.MESSAGE, 26) == L.MESSAGE))
    outs = {L.caesar_encrypt("A", k) for k in range(26)}
    checks.append(("26 keys give 26 different letters, 25 of them change the letter", len(outs) == 26))
    checks.append(("case and punctuation are kept", L.caesar_encrypt("Hello, World!", 3) == "Khoor, Zruog!"))
    checks.append(("all_shifts returns exactly 25 candidates", len(L.all_shifts("KHOOR")) == 25))
    checks.append(("one of the 25 candidates for KHOOR is HELLO", any(t == "HELLO" for _, t in L.all_shifts("KHOOR"))))
    ct3 = L.caesar_encrypt(L.MESSAGE, 3)
    checks.append(("brute force recovers key 3 automatically", L.best_shift(ct3) == 3))
    checks.append(("brute force recovers every key from 1 to 25",
                   all(L.best_shift(L.caesar_encrypt(L.MESSAGE, k)) == k for k in range(1, 26))))
    r = L.readability(L.MESSAGE)
    checks.append(("the note scores about 79 percent common letters", abs(r - 0.7917) < 0.001))
    sc = L.score_by_guess(ct3)
    wrong = max(s for g, s in enumerate(sc) if g != 3)
    checks.append(("right guess 79 percent beats the best wrong guess 67 percent",
                   abs(sc[3] - 0.7917) < 0.001 and abs(wrong - 0.6667) < 0.001))
    checks.append(("a flat alphabet scores 9/26 = 0.346 (35 percent)", abs(L.readability(L.ALPHABET) - 9 / 26) < 1e-12))
    checks.append(("hand calculation: GO with key 20 is AI (G=6, 6+20=26 wraps to 0; O=14, 34-26=8)", L.caesar_encrypt("GO", 20) == "AI"))
    checks.append(("key 13 is its own inverse: HELLO becomes URYYB and back", L.caesar_encrypt("HELLO", 13) == "URYYB" and L.caesar_encrypt("URYYB", 13) == "HELLO"))
    checks.append(("a string with no letters scores 0 (no division by zero)", L.readability("123 !?") == 0.0))
    checks.append(("trying all keys takes well under one second", L.time_to_try_all(ct3) < 1.0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
